In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Tue Nov  3 23:56:49 PST 2026


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, gbd_data
from ihme_cc_get_estimates import get_machinery_estimates

In [3]:
location = "india"
vehicle = "rice"

In [4]:
# Parameters
location = "ethiopia"
vehicle = "salt"


In [5]:
intervention_scenarios = config_utils.get_intervention_scenarios(location, vehicle)
intervention_scenarios

['intervention_25_nrv', 'intervention_100_nrv', 'intervention_45_ppm']

In [6]:
# Get most recent GBD year, used by calls in gbd_data
MOST_RECENT_GBD_YEAR = gbd_data.most_recent_year()
MOST_RECENT_GBD_YEAR

2023

In [7]:
ESTIMATION_YEAR = 2035

## Forecasted births and stillbirths

In [8]:
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate,
        "estimate",
        location.title(),
        # NOTE: Hardcoded (rather than using MOST_RECENT_GBD_YEAR) to ensure this is only updated manually,
        # since it *must* stay in sync with the denominator of the TFR ratio below
        years=2023,
    ).value

In [9]:
# Filter out lower and upper values, keep mean only
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr.loc[asfr>0]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  10.0       15.0     2023        2024        0.002928
                  15.0       20.0     2023        2024        0.047716
                  20.0       25.0     2023        2024        0.185350
                  25.0       30.0     2023        2024        0.181771
                                                                ...   
                  35.0       40.0     2023        2024        0.130724
                  40.0       45.0     2023        2024        0.062533
                  45.0       50.0     2023        2024        0.018075
                  50.0       55.0     2023        2024        0.001639
Name: value, Length: 9, dtype: float64

In [10]:
# TODO: Update this to forecasts based on GBD 2019 or 2021, pending getting these from the forecasting team
# Scale ASFR in each category down proportionally to the scale-down in
# total fertility rate (TFR) forecasted from GBD 2017
if location == "india":
    asfr_2035_to_2023_ratio = 1.49 / 1.87  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2035_to_2023_ratio = 4.02 / 4.91  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2035_to_2023_ratio = 2.81 / 4.00  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2035_to_2023_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  10.0       15.0     2023        2024        0.002057
                  15.0       20.0     2023        2024        0.033520
                  20.0       25.0     2023        2024        0.130208
                  25.0       30.0     2023        2024        0.127694
                                                                ...   
                  35.0       40.0     2023        2024        0.091834
                  40.0       45.0     2023        2024        0.043930
                  45.0       50.0     2023        2024        0.012698
                  50.0       55.0     2023        2024        0.001151
Name: value, Length: 9, dtype: float64

In [11]:
# Values now represent 2035 instead of 2023
# NOTE: This is kept hardcoded to ensure this is only updated manually, since
# it *must* stay in sync with the numerator of the TFR ratio above
asfr = (
    asfr.reset_index()
    .assign(year_start=2035, year_end=2036)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
Ethiopia  Female  0.000000   0.019178   2035        2036        0.000000
                  0.019178   0.076712   2035        2036        0.000000
                  0.076712   0.500000   2035        2036        0.000000
                  0.500000   1.000000   2035        2036        0.000000
                                                                  ...   
                  35.000000  40.000000  2035        2036        0.091834
                  30.000000  35.000000  2035        2036        0.127610
                  25.000000  30.000000  2035        2036        0.127694
                  20.000000  25.000000  2035        2036        0.130208
Name: value, Length: 50, dtype: float64

In [12]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import resolve_location
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [13]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # https://jira.ihme.washington.edu/browse/MIC-5204 to make this easier
    location_id = resolve_location(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        # NOTE: Using RELEASE_IDS.GBD_2023 returns an empty dataframe,
        # which we assume is because this forecast lags behind GBD by about 1 round --
        # this is the latest available as of 9/2/2026
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [14]:
with gbd_data.quiet_gbd_logs():
    pop = get_population_future(location.title(), ESTIMATION_YEAR).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
Ethiopia  Female  0.000000   0.019178    2035        2036         33892.747562
                  0.019178   0.076712    2035        2036        100984.609982
                  0.076712   0.500000    2035        2036                  NaN
                  0.500000   1.000000    2035        2036                  NaN
                                                                     ...      
          Male    80.000000  85.000000   2035        2036        254276.863715
                  85.000000  90.000000   2035        2036        117183.904618
                  90.000000  95.000000   2035        2036         41165.971745
                  95.000000  125.000000  2035        2036         11177.635038
Name: value, Length: 50, dtype: float64

In [15]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  0.076712   0.5      2035        2036       NaN
                  0.500000   1.0      2035        2036       NaN
                  1.000000   2.0      2035        2036       NaN
                  2.000000   5.0      2035        2036       NaN
          Male    0.076712   0.5      2035        2036       NaN
                  0.500000   1.0      2035        2036       NaN
                  1.000000   2.0      2035        2036       NaN
                  2.000000   5.0      2035        2036       NaN
Name: value, dtype: float64

In [16]:
pop = pop.fillna(0)

In [17]:
n_births = (pop * asfr).sum()
n_births

3682415.342716322

In [18]:
# NOTE: The stillbirth ratio (SBR) pulled from GBD here will be applied
# to the future population in the target year.
# The SBR does not vary much by year, so we just use data for the
# most recent GBD year (defined in YEAR above) and extrapolate that
# value forward to the target year.
with gbd_data.quiet_gbd_logs():
    sbr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio,
        "estimate",
        location.title(),
        years=MOST_RECENT_GBD_YEAR,
    ).value
sbr

location  year_start  year_end  parameter  
Ethiopia  2023        2024      lower_value    0.019651
                                mean_value     0.023079
                                upper_value    0.027940
Name: value, dtype: float64

In [19]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
).squeeze()
sbr

0.02307869896115

In [20]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

3.7674006978607917

## Fertility (technically birth-and-stillbirth) disparities

In [21]:
# TODO: Update this DHS data, and move it into 0100_data_prep? We already
# have some stuff in there to process DHS data.
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        {  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            1: 56_979,
            2: 50_335,
            3: 45_189,
            4: 42_611,
            5: 36_290,
        }
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        {  # Table 8.4 Perinatal mortality
            1: 7_712,
            2: 7_886,
            3: 7_139,
            4: 6_328,
            5: 5_558,
        }
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        {  # Table 8.4 Perinatal mortality
            1: 2_645,
            2: 2_516,
            3: 2_290,
            4: 2_018,
            5: 1_592,
        }
    )

dist_births_and_stillbirths_by_wealth.index.name = "wealth_quintile"

# TODO: Clarify variable names: What does the prefix s_ stand for in
# this notebook? Claude thinks it means "stratified," but that it is
# applied inconsistently and sometimes redundantly with the suffix
# _by_wealth.
s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

wealth_quintile
1    880570.344588
2    837623.813604
3    762384.154671
4    671830.228876
5    530006.800977
dtype: float64

In [22]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

wealth_quintile
1    900892.762485
2    856955.081441
3    779978.989070
4    687335.196482
5    542238.668384
dtype: float64

In [23]:
# Updated 2035 NTD deaths using forecasts based on GBD 2019/2021 data. Using GBD Foresight Visualization to replace get_outputs
# Population group: <1 year, both sexes; Metric: number, deaths; Cause: Neural Tube Defects

if location == "india":
    ntd_deaths = 3459.34 # http://ihmeuw.org/7sy3 
elif location == "nigeria":
    ntd_deaths = 4863.88 # http://ihmeuw.org/7sy4 
elif location == "ethiopia":
    ntd_deaths = 1627.84 # http://ihmeuw.org/7svr

In [24]:
ntd_death_rate_per_birth = ntd_deaths / n_births
10_000 * ntd_death_rate_per_birth

4.420576845628796

In [25]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

16.945544574910386

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [26]:
# TODO: Search for updated folate intake data
if location == "india":
    folate_intake_by_wealth = pd.Series(
        {
            1: 220,
            2: 220,
            3: 220,
            4: 220,
            5: 220,  # NRV is 400 mcg/day
        }
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    folate_intake_by_wealth = pd.Series(
        {
            1: 189,
            2: 198,
            3: 197,
            4: 203,
            5: 208,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    folate_intake_by_wealth = pd.Series(
        {
            1: 166,
            2: 152,
            3: 137,
            4: 350,
            5: 469,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?

folate_intake_by_wealth.index.name = "wealth_quintile"

In [27]:
# TODO: See if we can find better data here
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        {  # it includes wealth stratification, but has a very low threshold for insufficiency
            1: 1,  # so I am assuming that most everyone is in the danger zone for low folate
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        {  # can CHAMPS offer more detail?  Need to infer wealth somehow
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        {  # supplementation studies don't make this easy, but here is a guess
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )

s_dist_deaths_by_wealth /= (s_dist_deaths_by_wealth * s_births).sum() / s_births.sum()
s_dist_deaths_by_wealth.index.name = "wealth_quintile"
s_dist_deaths_by_wealth

wealth_quintile
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
dtype: float64

In [28]:
s_ntd_death_rate_per_birth = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate_per_birth

wealth_quintile
1    4.420577
2    4.420577
3    4.420577
4    4.420577
5    4.420577
dtype: float64

In [29]:
s_ntd_death_count = s_ntd_death_rate_per_birth * s_births
s_ntd_death_count

wealth_quintile
1    389.262888
2    370.278044
3    337.017774
4    296.987715
5    234.293579
dtype: float64

In [30]:
assert np.isclose(s_ntd_death_count.sum(), ntd_deaths)

In [31]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

wealth_quintile
1    1102.911515
2    1049.121123
3     954.883693
4     841.465194
5     663.831808
dtype: float64

In [32]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

wealth_quintile
1    1492.174403
2    1419.399167
3    1291.901468
4    1138.452909
5     898.125387
dtype: float64

In [33]:
# NOTE: NTD risk here means risk of having an "NTD-affected pregnancy",
# which is a stillbirth due to NTD, OR a birth with NTD (not necessarily fatal!)
# See Kirke 1993 ("Maternal plasma folate and vitamin B12 are independent risk factors for neural tube defects")
# where it says: "Early foetal
# deaths (<23 weeks gestation) attributable to NTDs
# were excluded because of the incomplete ascertain-
# ment of such cases and the difficulty of obtaining a
# valid control group."
# This implies that late foetal deaths, roughly equivalent to stillbirths,
# are included.
def rbc_folate_from_ntd_risk(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc

# The inverse of the above
def ntd_probability_from_rbc_folate(rbc, method):
    ln_rbc = np.log(rbc)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds) / (1 + np.exp(ln_odds))
    return p

for ntd_risk in [0.01, 0.05, 0.1]:
    assert np.isclose(ntd_probability_from_rbc_folate(rbc_folate_from_ntd_risk(ntd_risk, "daly"), "daly"), ntd_risk, rtol=0, atol=1e-10)
    assert np.isclose(ntd_probability_from_rbc_folate(rbc_folate_from_ntd_risk(ntd_risk, "crider"), "crider"), ntd_risk, rtol=0, atol=1e-10)

In [34]:
# NTD incident cases / NTD deaths in <1 year olds
# (all of GBD's incident cases are those who survived birth)

with gbd_data.quiet_gbd_logs():
    ntd_incident_cases_2023 = get_machinery_estimates(
        entity="cause",
        entity_id=int(gbd_mapping.causes.neural_tube_defects.gbd_id),
        release_id=RELEASE_IDS.GBD_2023,
        year_id=MOST_RECENT_GBD_YEAR,
        location_id=[resolve_location(location.title())],
        age_group_id=28, # < 1 year
        sex_id=3, # Both sexes
        measure_id=6, # Incidence
        metric_id=1, # Number
    ).rename(columns={"point_estimate": "val"})
    
    ntd_deaths_2023 = get_machinery_estimates(
        entity="cause",
        entity_id=int(gbd_mapping.causes.neural_tube_defects.gbd_id),
        release_id=RELEASE_IDS.GBD_2023,
        year_id=MOST_RECENT_GBD_YEAR,
        location_id=[resolve_location(location.title())],
        age_group_id=28, # < 1 year
        sex_id=3, # Both sexes
        measure_id=1, # Deaths
        metric_id=1, # Number
    ).rename(columns={"point_estimate": "val"})

assert len(ntd_incident_cases_2023) == 1
ntd_incident_cases_2023 = ntd_incident_cases_2023.iloc[0]['val']
assert len(ntd_deaths_2023) == 1
ntd_deaths_2023 = ntd_deaths_2023.iloc[0]['val']

# The ratio of live birth cases (NTD cases that were born alive)
# to NTD deaths
ntd_live_birth_case_to_death_ratio = ntd_incident_cases_2023 / ntd_deaths_2023
ntd_live_birth_case_to_death_ratio

2.0470023425980672

In [35]:
s_ntd_live_birth_cases = ntd_live_birth_case_to_death_ratio * s_ntd_death_count
s_ntd_live_birth_cases

wealth_quintile
1    796.822043
2    757.960023
3    689.876173
4    607.934549
5    479.599506
dtype: float64

In [36]:
s_ntd_affected_pregnancies = s_ntd_stillbirth_count + s_ntd_live_birth_cases

In [37]:
ntd_affected_pregnancy_risk = (
    s_ntd_affected_pregnancies / s_births_and_stillbirths_by_wealth
)
ntd_affected_pregnancy_risk

wealth_quintile
1    0.002109
2    0.002109
3    0.002109
4    0.002109
5    0.002109
dtype: float64

In [38]:
s_ntd_death_or_stillbirth_count / s_births

wealth_quintile
1    0.001695
2    0.001695
3    0.001695
4    0.001695
5    0.001695
dtype: float64

In [39]:
rbc_folate_from_ntd_risk(ntd_affected_pregnancy_risk, "daly")

wealth_quintile
1    607.96779
2    607.96779
3    607.96779
4    607.96779
5    607.96779
dtype: float64

In [40]:
rbc_folate_from_ntd_risk(
    ntd_affected_pregnancy_risk, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9
# NOTE: It's a bit hard to compare, due to units issues. That table reports
# proportions under 151 ng/ml. In our units (nmol/L), that is ~342.
# https://www.wolframalpha.com/input?i=151+ng%2Fml+of+folate+to+nmol%2FL

wealth_quintile
1    550.874406
2    550.874406
3    550.874406
4    550.874406
5    550.874406
dtype: float64

In [41]:
pop

location  sex     age_start  age_end     year_start  year_end
Ethiopia  Female  0.000000   0.019178    2035        2036         33892.747562
                  0.019178   0.076712    2035        2036        100984.609982
                  0.076712   0.500000    2035        2036             0.000000
                  0.500000   1.000000    2035        2036             0.000000
                                                                     ...      
          Male    80.000000  85.000000   2035        2036        254276.863715
                  85.000000  90.000000   2035        2036        117183.904618
                  90.000000  95.000000   2035        2036         41165.971745
                  95.000000  125.000000  2035        2036         11177.635038
Name: value, Length: 50, dtype: float64

In [42]:
s_pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
s_pop = s_pop.set_index([c for c in s_pop.columns if c != "value"])
s_pop

value
sex    age_start age_end    pregnant     wealth_quintile             
Female 0.0       0.019178   not_pregnant 1                8110.649607
                                         2                7372.711095
                                         3                6970.098109
                                         4                6250.761973
...                                                               ...
Male   95.0      125.000000 not_pregnant 2                1527.072977
                                         3                1585.027440
                                         4                1677.097974
                                         5                1469.804071

[285 rows x 1 columns]

In [43]:
# WRA only
s_pop = s_pop[
    (s_pop.index.get_level_values("sex") == "Female")
    & (s_pop.index.get_level_values("age_start") >= 15)
    & (s_pop.index.get_level_values("age_end") <= 50)
].copy()

In [44]:
s_daily_vehicle = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/amount/mean/{location}.csv"
)
assert (s_daily_vehicle.vehicle_name == vehicle).all()
s_daily_vehicle = s_daily_vehicle.drop(columns=["vehicle_name"])
s_daily_vehicle = s_daily_vehicle.set_index(
    [c for c in s_daily_vehicle.columns if c != "value"]
).value
s_daily_vehicle

wealth_quintile  sex     age_start  age_end
1                Female  15         50         7.084152
2                Female  15         50         7.235241
3                Female  15         50         7.704005
4                Female  15         50         7.157051
5                Female  15         50         6.504427
Name: value, dtype: float64

In [45]:
from lsff_utils import data_processing

In [46]:
s_daily_vehicle = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, s_daily_vehicle)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
s_daily_vehicle

wealth_quintile
1    7.084152
2    7.235241
3    7.704005
4    7.157051
5    6.504427
Name: value, dtype: float64

In [47]:
any_consumers = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
assert (any_consumers.vehicle_name == vehicle).all()
any_consumers = any_consumers.drop(columns=["vehicle_name"])
any_consumers = any_consumers.set_index(
    [c for c in any_consumers.columns if c != "value"]
).value
any_consumers

wealth_quintile  sex     age_start  age_end
1                Female  15         50         0.98
2                Female  15         50         0.98
3                Female  15         50         0.98
4                Female  15         50         0.98
5                Female  15         50         0.98
Name: value, dtype: float64

In [48]:
any_consumers = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, any_consumers)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
any_consumers

wealth_quintile
1    0.98
2    0.98
3    0.98
4    0.98
5    0.98
Name: value, dtype: float64

In [49]:
s_daily_vehicle_among_consumers = s_daily_vehicle / any_consumers
s_daily_vehicle_among_consumers

wealth_quintile
1    7.228727
2    7.382899
3    7.861230
4    7.303113
5    6.637170
Name: value, dtype: float64

In [50]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.0

In [51]:
intervention_concentration_mcg_per_gram = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.drop(columns=["vehicle_name"])
    .set_index("scenario")
    .value
)
intervention_concentration_mcg_per_gram

scenario
intervention_25_nrv     12.8
intervention_100_nrv    51.3
intervention_45_ppm     45.0
Name: value, dtype: float64

In [52]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()

In [53]:
if location == "india" and vehicle == "rice":
    # Confusingly, our baseline scenario (our best guess about the present)
    # is *not* a good guess about 2019-2020 (which is when our baseline folate estimate is from),
    # because this program has rolled out entirely since then:
    # In the phase-I of the roll out, the fortified rice was introduced in the social welfare schemes such as Integrated Child Development Scheme (ICDS)
    # and Pradhan Mantri Poshan Shakti Nirman (PM POSHAN, earlier known as the National Program of Mid-Day Meal in Schools)
    # throughout India during 2021–22 [18].
    # Phase-II has covered aspirational and high burden districts for anemia (total 291 districts) under Public Distribution System (PDS) and other welfare schemes,
    # in addition to Phase-I districts, by March 2023 [18].
    # All the remaining districts in India will be covered in Phase III by March 2024 [19].
    # ~ https://pmc.ncbi.nlm.nih.gov/articles/PMC11305529/
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline.assign(value=0)
else:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline

In [54]:
df_eff_fort_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()
df_eff_fort_intervention = df_eff_fort_intervention.drop(columns=["vehicle_name"])
df_eff_fort_intervention

,wealth_quintile,sex,value,scenario
0,1,Female,0.787998,intervention_25_nrv
1,1,Male,0.787998,intervention_25_nrv
2,2,Female,0.787998,intervention_25_nrv
3,2,Male,0.787998,intervention_25_nrv
...,...,...,...,...
6,4,Female,0.787998,intervention_45_ppm
7,4,Male,0.787998,intervention_45_ppm
8,5,Female,0.787998,intervention_45_ppm
9,5,Male,0.787998,intervention_45_ppm


In [55]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,1,1.019154e+06
41,Female,15.0,20.0,2,1.154405e+06
42,Female,15.0,20.0,3,1.160923e+06
43,Female,15.0,20.0,4,1.239019e+06
...,...,...,...,...,...
71,Female,45.0,50.0,2,3.496229e+05
72,Female,45.0,50.0,3,3.891729e+05
73,Female,45.0,50.0,4,4.024039e+05
74,Female,45.0,50.0,5,4.670795e+05


In [56]:
if "sex" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.sex == "Female")
    ]

if "age_start" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.age_start >= 15)
        & (df_eff_fort_baseline_2019_2020.age_end <= 50)
    ]

In [57]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [58]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population) * (
        1
        if "scenario" not in effective_fort.columns
        else effective_fort.scenario.nunique()
    )
    group_cols = [c for c in ["scenario", "wealth_quintile"] if c in merged]
    return merged.groupby(group_cols).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [59]:
df_eff_fort_baseline_2019_2020 = aggregate_using_population(
    df_eff_fort_baseline_2019_2020
)
df_eff_fort_baseline_2019_2020

   vehicle_name  wealth_quintile  value_fort  index     sex  age_start  \
0          salt                1         0.0     40  Female       15.0   
1          salt                1         0.0     45  Female       20.0   
2          salt                1         0.0     50  Female       25.0   
3          salt                1         0.0     55  Female       30.0   
..          ...              ...         ...    ...     ...        ...   
31         salt                5         0.0     59  Female       30.0   
32         salt                5         0.0     64  Female       35.0   
33         salt                5         0.0     69  Female       40.0   
34         salt                5         0.0     74  Female       45.0   

    age_end     value_pop  
0      20.0  1.019154e+06  
1      25.0  9.466980e+05  
2      30.0  8.116082e+05  
3      35.0  7.923943e+05  
..      ...           ...  
31     35.0  9.194147e+05  
32     40.0  7.731326e+05  
33     45.0  6.274605e+05  
34     

/tmp/ipykernel_2786423/3772084468.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby(group_cols).apply(


wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [60]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

   vehicle_name  wealth_quintile  value_fort  index     sex  age_start  \
0          salt                1         0.0     40  Female       15.0   
1          salt                1         0.0     45  Female       20.0   
2          salt                1         0.0     50  Female       25.0   
3          salt                1         0.0     55  Female       30.0   
..          ...              ...         ...    ...     ...        ...   
31         salt                5         0.0     59  Female       30.0   
32         salt                5         0.0     64  Female       35.0   
33         salt                5         0.0     69  Female       40.0   
34         salt                5         0.0     74  Female       45.0   

    age_end     value_pop  
0      20.0  1.019154e+06  
1      25.0  9.466980e+05  
2      30.0  8.116082e+05  
3      35.0  7.923943e+05  
..      ...           ...  
31     35.0  9.194147e+05  
32     40.0  7.731326e+05  
33     45.0  6.274605e+05  
34     

/tmp/ipykernel_2786423/3772084468.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby(group_cols).apply(


wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [61]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

     wealth_quintile     sex  value_fort             scenario  index  \
0                  1  Female    0.787998  intervention_25_nrv     40   
1                  1  Female    0.787998  intervention_25_nrv     45   
2                  1  Female    0.787998  intervention_25_nrv     50   
3                  1  Female    0.787998  intervention_25_nrv     55   
..               ...     ...         ...                  ...    ...   
101                5  Female    0.787998  intervention_45_ppm     59   
102                5  Female    0.787998  intervention_45_ppm     64   
103                5  Female    0.787998  intervention_45_ppm     69   
104                5  Female    0.787998  intervention_45_ppm     74   

     age_start  age_end     value_pop  
0         15.0     20.0  1.019154e+06  
1         20.0     25.0  9.466980e+05  
2         25.0     30.0  8.116082e+05  
3         30.0     35.0  7.923943e+05  
..         ...      ...           ...  
101       30.0     35.0  9.194147e+05  

/tmp/ipykernel_2786423/3772084468.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby(group_cols).apply(


scenario              wealth_quintile
intervention_100_nrv  1                  0.787998
                      2                  0.787998
                      3                  0.787998
                      4                  0.787998
                                           ...   
intervention_45_ppm   2                  0.787998
                      3                  0.787998
                      4                  0.787998
                      5                  0.787998
Length: 15, dtype: float64

In [62]:
RBC_baseline = rbc_folate_from_ntd_risk(ntd_affected_pregnancy_risk, "crider")
RBC_baseline

wealth_quintile
1    550.874406
2    550.874406
3    550.874406
4    550.874406
5    550.874406
dtype: float64

In [63]:
# Fortification folic acid needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [64]:
# TODO: Determine whether we still want the same fortification
# scenarios, and update baseline values
# Delete fortification effect baked into our baseline folate estimate.
# In non-India locations, this is going to be zero.
# For India, our current source for baseline folate is very rough,
# but it does appear to be from before the fortification program (2019-2020).
s_zero_folate = folate_intake_by_wealth - (
    df_eff_fort_baseline_2019_2020
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)

In [65]:
s_baseline_folate = s_zero_folate + (
    df_eff_fort_baseline
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_baseline_folate

wealth_quintile
1    166.0
2    152.0
3    137.0
4    350.0
5    469.0
dtype: float64

In [66]:
s_intervention_folate = s_zero_folate + (
    df_eff_fort_intervention
    * s_daily_vehicle_among_consumers
    * intervention_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_intervention_folate

scenario              wealth_quintile
intervention_100_nrv  1                  662.767777
                      2                  659.362714
                      3                  677.234235
                      4                  851.879733
                                            ...    
intervention_45_ppm   2                  597.055013
                      3                  610.889680
                      4                  790.245380
                      5                  869.101094
Length: 15, dtype: float64

In [67]:
zero_folate_pct_decrease = (
    folate_intake_by_wealth - s_zero_folate
) / folate_intake_by_wealth

In [68]:
baseline_folate_pct_increase_from_zero = (
    s_baseline_folate - s_zero_folate
) / folate_intake_by_wealth
baseline_folate_pct_increase_from_zero

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [69]:
intevention_folate_pct_increase_from_zero = (
    s_intervention_folate - s_zero_folate
) / folate_intake_by_wealth
intevention_folate_pct_increase_from_zero

scenario              wealth_quintile
intervention_100_nrv  1                  2.992577
                      2                  3.337913
                      3                  3.943316
                      4                  1.433942
                                           ...   
intervention_45_ppm   2                  2.927994
                      3                  3.459049
                      4                  1.257844
                      5                  0.853094
Length: 15, dtype: float64

In [70]:
# Previous effect size, used in 2024 results:
# https://pubmed.ncbi.nlm.nih.gov/25867949/
# Marchetta et al. 2016 says "a 6% (95% Credible Interval (CrI): 4%, 9%) increase in red blood cell (RBC) folate concentration... can occur for every 10% increase in natural food folate intake"
INTAKE_TO_RBC_FOLATE_ELASTICITY_MARCHETTA = np.log(1.06) / np.log(1.10)

# New effect size for 2026 results:
# Crider et al. 2019 (Nutrients 11:71) + Tessema et al. 2026 (AJCN 123:101204).
# RBC folate responds to TOTAL folate intake T as a power law, RBC = A * T**e,
# so a proportional change in intake gives a proportional change in RBC folate.
# e = 0.43 for folic acid, replacing Marchetta's 0.6, which was estimated on
# natural food folate only. Sensitivity range 0.40 - 0.46.
INTAKE_TO_RBC_FOLATE_ELASTICITY = 0.43

def rbc_folate_ratio(intake_from, intake_to):
    """RBC folate ratio between two total-intake levels, both mcg DFE/day."""
    return (intake_to / intake_from) ** INTAKE_TO_RBC_FOLATE_ELASTICITY

# RBC_baseline from cell 64 corresponds to baseline intake stored in
# folate_intake_by_wealth. Step back to the zero-fortification
# counterfactual.
RBC_zero = RBC_baseline * rbc_folate_ratio(folate_intake_by_wealth, s_zero_folate)
RBC_zero


wealth_quintile
1    550.874406
2    550.874406
3    550.874406
4    550.874406
5    550.874406
dtype: float64

In [71]:
RBC_baseline = RBC_zero * rbc_folate_ratio(s_zero_folate, s_baseline_folate)
RBC_baseline

wealth_quintile
1    550.874406
2    550.874406
3    550.874406
4    550.874406
5    550.874406
dtype: float64

In [72]:
RBC_intervention = RBC_zero * rbc_folate_ratio(s_zero_folate, s_intervention_folate)
RBC_intervention

scenario              wealth_quintile
intervention_100_nrv  1                   999.059866
                      2                  1035.340822
                      3                  1095.167331
                      4                   807.543637
                                            ...     
intervention_45_ppm   2                   992.078465
                      3                  1047.675409
                      4                   781.881579
                      5                   718.205126
Length: 15, dtype: float64

In [73]:
# NOTE: All rates here are per birth!
s_ntd_affected_pregnancy_rate_zero = ntd_probability_from_rbc_folate(RBC_zero, "crider")
10_000 * s_ntd_affected_pregnancy_rate_zero

wealth_quintile
1    21.087233
2    21.087233
3    21.087233
4    21.087233
5    21.087233
dtype: float64

In [74]:
s_ntd_affected_pregnancy_rate_baseline = ntd_probability_from_rbc_folate(RBC_baseline, "crider")
10_000 * s_ntd_affected_pregnancy_rate_baseline

wealth_quintile
1    21.087233
2    21.087233
3    21.087233
4    21.087233
5    21.087233
dtype: float64

In [75]:
s_ntd_affected_pregnancy_rate_intervention = ntd_probability_from_rbc_folate(RBC_intervention, "crider")
10_000 * s_ntd_affected_pregnancy_rate_intervention

scenario              wealth_quintile
intervention_100_nrv  1                   7.675148
                      2                   7.223876
                      3                   6.566345
                      4                  11.017038
                                           ...    
intervention_45_ppm   2                   7.767121
                      3                   7.079991
                      4                  11.638054
                      5                  13.443708
Length: 15, dtype: float64

In [76]:
s_ntd_affected_pregnancies_zero = (
    s_ntd_affected_pregnancy_rate_zero * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_zero

wealth_quintile
1    1899.733558
2    1807.081146
3    1644.759867
4    1449.399743
5    1143.431313
dtype: float64

In [77]:
s_ntd_affected_pregnancies_baseline = (
    s_ntd_affected_pregnancy_rate_baseline * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_baseline

wealth_quintile
1    1899.733558
2    1807.081146
3    1644.759867
4    1449.399743
5    1143.431313
dtype: float64

In [78]:
s_ntd_affected_pregnancies_intervention = (
    s_ntd_affected_pregnancy_rate_intervention * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_intervention

scenario              wealth_quintile
intervention_100_nrv  1                  691.448522
                      2                  619.053700
                      3                  512.161116
                      4                  757.239813
                                            ...    
intervention_45_ppm   2                  665.607412
                      3                  552.224437
                      4                  799.924445
                      5                  728.969857
Length: 15, dtype: float64

In [79]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_affected_pregnancies_zero.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="zero")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_affected_pregnancies_baseline.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        *[
            s_ntd_affected_pregnancies_intervention.loc[intervention_scenario]
            .rename("value")
            .to_frame()
            .assign(entity="ntd", scenario=intervention_scenario)
            .set_index(["entity", "scenario"], append=True)
            .value
            for intervention_scenario in intervention_scenarios
        ],
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario           
1                ntd     zero                   1899.733558
2                ntd     zero                   1807.081146
3                ntd     zero                   1644.759867
4                ntd     zero                   1449.399743
                                                   ...     
2                ntd     intervention_45_ppm     665.607412
3                ntd     intervention_45_ppm     552.224437
4                ntd     intervention_45_ppm     799.924445
5                ntd     intervention_45_ppm     728.969857
Name: value, Length: 25, dtype: float64

In [80]:
(
    ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "baseline"
    ].droplevel("scenario")
    - ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "intervention"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd      NaN
2                ntd      NaN
3                ntd      NaN
4                ntd      NaN
5                ntd      NaN
Name: value, dtype: float64

In [81]:
(
    ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "zero"
    ].droplevel("scenario")
    - ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "baseline"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       0.0
2                ntd       0.0
3                ntd       0.0
4                ntd       0.0
5                ntd       0.0
Name: value, dtype: float64

In [82]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [83]:
ntd_deaths_and_stillbirths_by_scenario = ntd_cases_by_scenario * (
    s_ntd_death_or_stillbirth_count / s_ntd_affected_pregnancies
)

In [84]:
(
    ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "baseline"
    ].droplevel("scenario")
    - ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "intervention"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd      NaN
2                ntd      NaN
3                ntd      NaN
4                ntd      NaN
5                ntd      NaN
dtype: float64

In [85]:
(
    ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "zero"
    ].droplevel("scenario")
    - ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "baseline"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       0.0
2                ntd       0.0
3                ntd       0.0
4                ntd       0.0
5                ntd       0.0
dtype: float64

In [86]:
# For calculating YLLs
with gbd_data.quiet_gbd_logs():
    tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [87]:
# NOTE: Treating stillbirths as a death!
yll_per_stillbirth_or_death = float(tmrle.iloc[0])
yll_per_stillbirth_or_death

/tmp/ipykernel_2786423/689930244.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  yll_per_stillbirth_or_death = float(tmrle.iloc[0])


89.95803974533831

In [88]:
ylls_by_scenario = (
    ntd_deaths_and_stillbirths_by_scenario * yll_per_stillbirth_or_death
).rename("value")
ylls_by_scenario

wealth_quintile  entity  scenario           
1                ntd     zero                   134233.084212
2                ntd     zero                   127686.366683
3                ntd     zero                   116216.923571
4                ntd     zero                   102412.992038
                                                    ...      
2                ntd     intervention_45_ppm     47031.087824
3                ntd     intervention_45_ppm     39019.571506
4                ntd     intervention_45_ppm     56521.781688
5                ntd     intervention_45_ppm     51508.208531
Name: value, Length: 25, dtype: float64

In [89]:
path = f"./results/{location}/{vehicle}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)